# Trojan Verification

**CSAW 2026 AI Hardware Attack Challenge — The Bitstream Bandits**

This notebook independently verifies the three hardware Trojans generated under
`notebooks/trojans/` by **simulating** each one against the recovered
cryptographic accelerator (the generation notebooks only compile-check). For
every variant it establishes two things:

1. **Normal operation is preserved** — a full encrypt and decrypt still return
   the supplied golden vectors, so the cryptographic datapath is untouched when
   the Trojan is dormant.
2. **The attack works as specified** — the documented trigger activates the
   payload, a one-bit-off *near miss* does **not**, and (where relevant) the
   device recovers.

| Variant | Selected RTL | Trigger | Expected payload |
| --- | --- | --- | --- |
| Functionality corruption | `attempt_009.sv` | SPI word `0xA6F03C59` | next transaction's 32 MISO bits inverted (one-shot) |
| Denial of service | `attempt_007.sv` | SPI word `0xBAADF00D` | BUSY + ICE_LED latched high until reset |
| Performance degradation | `attempt_010.sv` | 4th START after reset | BUSY extended by 8 SCK cycles |

Golden vectors: `encrypt(0x59C359C3) = 0x9CD84392`, `decrypt(0x9CD84392) = 0x59C359C3`.

The harness lives in `tb/trojan_verification/` and drives the accelerator over
SPI exactly like the RP2040 reference application
(`inputs/spi_ice40_crypto_ip_test.py`): mode 0, MSB-first, 32-bit words, with
SCK as the sole system clock. Simulation uses Icarus Verilog and the iCE40
primitive models from OSS CAD Suite. No network or AI gateway is required.

## 1. Setup

Locate the repository and toolchain, and prepare the renamed core.

In [ ]:
from pathlib import Path
import os, re, glob, json, shutil, subprocess, datetime


def find_repo_root(markers, g):
    """Find the repo root from the notebook file (VS Code) or the cwd, robust
    to wherever the Jupyter kernel was started."""
    starts = []
    for var in ("__vsc_ipynb_file__", "__file__", "__session__"):
        val = g.get(var)
        if val:
            try:
                starts.append(Path(val).resolve().parent)
            except Exception:
                pass
    starts.append(Path.cwd().resolve())
    for start in starts:
        for d in (start, *start.parents):
            for sub in ("", "aha2026-dp1"):
                cand = d / sub if sub else d
                if all((cand / m).exists() for m in markers):
                    return cand
    raise FileNotFoundError(
        f"Could not locate the aha2026-dp1 repository from {Path.cwd()}; "
        "open this notebook inside the repository.")


def discover_toolchain():
    """Find iverilog, vvp and the iCE40 cells_sim.v from OSS CAD Suite."""
    cands = []
    if os.environ.get("OSS_CAD_SUITE"):
        cands.append(Path(os.environ["OSS_CAD_SUITE"]).expanduser())
    cands.append(Path("~/.local/opt/oss-cad-suite").expanduser())
    cands += [Path(p) for p in sorted(
        glob.glob(str(Path("~/.local/opt/oss-cad-suite*").expanduser())), reverse=True)]
    suite = next((c.resolve() for c in cands if (c / "bin/iverilog").is_file()), None)

    def pick(name):
        if suite and (suite / "bin" / name).is_file():
            return str(suite / "bin" / name)
        found = shutil.which(name)
        if not found:
            raise FileNotFoundError(f"{name} not found; install OSS CAD Suite or set OSS_CAD_SUITE")
        return found

    tools = {n: pick(n) for n in ("iverilog", "vvp")}
    env_cells = os.environ.get("ICE40_CELLS_SIM")
    if env_cells and Path(env_cells).is_file():
        tools["cells"] = str(Path(env_cells).resolve())
    elif suite and (suite / "share/yosys/ice40/cells_sim.v").is_file():
        tools["cells"] = str(suite / "share/yosys/ice40/cells_sim.v")
    else:
        hit = sorted(glob.glob(str(
            Path("~/.local/opt/oss-cad-suite*/share/yosys/ice40/cells_sim.v").expanduser())), reverse=True)
        if not hit:
            raise FileNotFoundError("cells_sim.v not found; set ICE40_CELLS_SIM")
        tools["cells"] = hit[0]
    return tools


ROOT = find_repo_root(("recovery/recovered_sim.v", "tb/trojan_verification"), globals())
TOOLS = discover_toolchain()
TBDIR = ROOT / "tb/trojan_verification"
BUILD = ROOT / "build/trojan_verification"
EVID = ROOT / "results/trojan_verification"
BUILD.mkdir(parents=True, exist_ok=True)
EVID.mkdir(parents=True, exist_ok=True)

# Rename the recovered core module: recovered -> recovered_original, so a Trojan
# wrapper (itself named `recovered`) can instantiate the unmodified core.
CORE = BUILD / "recovered_original.v"
_src = (ROOT / "recovery/recovered_sim.v").read_text()
_renamed, _n = re.subn(r"\bmodule\s+recovered\s*\(", "module recovered_original(", _src, count=1)
assert _n == 1, "expected exactly one 'module recovered(' in recovered_sim.v"
CORE.write_text(_renamed)

CLEAN_CORE = ROOT / "recovery/recovered_sim.v"
TROJANS = {
    "functionality": {"rtl": ROOT / "rtl/trojans/functionality/attempt_009.sv", "trigger": "0xA6F03C59"},
    "dos":           {"rtl": ROOT / "rtl/trojans/dos/attempt_007.sv",           "trigger": "0xBAADF00D"},
    "performance":   {"rtl": ROOT / "rtl/trojans/performance/attempt_010.sv",   "trigger": "4th START"},
}
RESULTS = {}

print("Repository :", ROOT)
print("iverilog   :", TOOLS["iverilog"])
print("vvp        :", TOOLS["vvp"])
print("iCE40 cells:", TOOLS["cells"])
print("Core copy  :", CORE.relative_to(ROOT), "(module renamed to recovered_original)")
for _name, _info in TROJANS.items():
    print(f"  {_name:13s} -> {_info['rtl'].relative_to(ROOT)}   trigger = {_info['trigger']}")

## 2. Verification harness

The reusable pieces under `tb/trojan_verification/`:

- **`crypto_dut.sv`** — exposes the accelerator (`recovered`) through readable
  processor-side names. The same wrapper drives the clean core *or* any Trojan,
  because every Trojan keeps the original port list.
- **`spi_operations.svh`** — the SPI controller tasks (reset, 32-bit transfer,
  START pulse, completion polling), reused from the project's SPI testbench
  workflow.
- **`baseline_tb.sv`** — full encrypt + decrypt golden-vector check (normal op).
- **`functionality_tb.sv`, `dos_tb.sv`, `performance_tb.sv`** — the per-Trojan
  trigger / payload / non-trigger checks.

`run_case()` compiles one DUT + testbench with Icarus Verilog and runs it under
`vvp`. A case passes only when the testbench prints `SUMMARY PASS` and nothing
prints `FAIL`.

In [ ]:
def run_case(label, top, tb, dut_sources):
    """Compile (iverilog) and simulate (vvp) one DUT + testbench."""
    out = BUILD / f"{label}.vvp"
    cmd = [TOOLS["iverilog"], "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
           "-s", top, "-I", str(TBDIR), "-o", str(out)]
    cmd += [str(p) for p in dut_sources] + [TOOLS["cells"], str(TBDIR / tb)]
    comp = subprocess.run(cmd, capture_output=True, text=True, cwd=BUILD)
    if comp.returncode != 0:
        r = {"label": label, "ok": False, "stage": "compile", "log": comp.stdout + comp.stderr}
    else:
        sim = subprocess.run([TOOLS["vvp"], str(out)], capture_output=True, text=True, cwd=BUILD)
        log = sim.stdout + sim.stderr
        r = {"label": label, "ok": sim.returncode == 0 and "SUMMARY PASS" in log and "FAIL" not in log,
             "stage": "sim", "log": log}
    RESULTS[label] = r
    return r


def show(result, keep=r"PASS|FAIL|SUMMARY|FUNC |PERF |DOS |extra cycles"):
    print(f"[{'PASS' if result['ok'] else 'FAIL'}] {result['label']}")
    for line in result["log"].splitlines():
        if re.search(keep, line):
            print("    " + line)
    if not result["ok"] and result["stage"] == "compile":
        print(result["log"][-1500:])


def dut_sources(name):
    """Trojan DUT = wrapper + Trojan RTL + renamed original core."""
    return [TBDIR / "crypto_dut.sv", TROJANS[name]["rtl"], CORE]

## 3. Normal operation preserved (all variants)

Run the clean recovered core as a reference, then each Trojan, through
`baseline_tb` (which never sends a trigger word). Every design must return the
golden ciphertext/plaintext — evidence that the Trojans leave the cryptographic
datapath intact during normal use.

In [ ]:
show(run_case("clean_core.normal", "baseline_tb", "baseline_tb.sv",
              [TBDIR / "crypto_dut.sv", CLEAN_CORE]))
for _name in TROJANS:
    show(run_case(f"{_name}.normal", "baseline_tb", "baseline_tb.sv", dut_sources(_name)))


def grab(label, key):
    m = re.search(rf"PASS {re.escape(key)} value=([0-9a-fA-F]+)", RESULTS[label]["log"])
    return m.group(1) if m else "--------"

print("\n  design                 enc.echo  encrypt   decrypt   result")
print("  " + "-" * 60)
for _label in ["clean_core.normal", "functionality.normal", "dos.normal", "performance.normal"]:
    print(f"  {_label:21s}  {grab(_label,'enc.echo')}  {grab(_label,'enc.result')}  "
          f"{grab(_label,'dec.result')}  {'PASS' if RESULTS[_label]['ok'] else 'FAIL'}")

## 4. Functionality-corruption Trojan — trigger & payload

The SPI port is a 32-bit loopback, so the word read during transaction *N*
equals the word shifted in during transaction *N-1*. That gives a known MISO
value to watch:

- the trigger transaction (`0xA6F03C59`) itself is **unaffected**;
- the single following transaction returns `~0xA6F03C59 = 0x590FC3A6` (inverted);
- the next transaction is back to normal (one-shot payload);
- a near miss `0xA6F03C58` (one bit off) causes **no** inversion.

In [ ]:
show(run_case("functionality.attack", "functionality_tb", "functionality_tb.sv", dut_sources("functionality")))

## 5. Denial-of-service Trojan — trigger & payload

The trigger word `0xBAADF00D` forces BUSY and ICE_LED high and keeps them there
— across further SCK pulses and even with no clock at all — until a synchronous
reset. After reset the core encrypts correctly again. A near miss `0xBAADF00C`
never latches BUSY.

In [ ]:
show(run_case("dos.attack", "dos_tb", "dos_tb.sv", dut_sources("dos")))

## 6. Performance-degradation Trojan — trigger & payload

Six START operations are issued after a single reset so the Trojan's internal
START counter advances 1..6. The testbench measures how many SCK cycles BUSY
stays high after each START: normal operations last 3 cycles, while the **4th**
operation is held for 3 + 8 = 11 cycles.

In [ ]:
_r = run_case("performance.attack", "performance_tb", "performance_tb.sv", dut_sources("performance"))
show(_r)
_m = re.search(r"PERF DURATIONS op1\.\.op6 = (.+)", _r["log"])
if _m:
    _d = _m.group(1).split()
    print("\n  operation   1   2   3   4   5   6")
    print("  BUSY cycles " + "".join(f"{v:>4}" for v in _d) + "   <- op4 extended by 8")

## 7. Waveform evidence

Each testbench dumps a VCD. The cell below parses those VCDs and renders the
trigger/payload windows as timing diagrams, saved to `docs/waveforms/` and
embedded in README Section 5. They show each payload mechanism directly:

- **Functionality** — during the payload transaction `MISO` is the exact
  complement of the core's `orig_miso` (the inverted echo `0x590FC3A6`).
- **DoS** — `trigger`, `BUSY` and `ICE_LED` latch high and persist (even with
  SCK stopped) until `RST_N` clears them.
- **Performance** — on the 4th START the core's `orig_busy` ends at cycle 3
  while external `BUSY` is held to cycle 11, with `ext_cnt` counting the
  8-cycle extension.

In [ ]:
from bisect import bisect_right
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    from IPython.display import Image, display
    _HAVE_MPL = True
except ImportError as _exc:
    _HAVE_MPL = False
    print("matplotlib/IPython not available in this kernel; keeping committed PNGs:", _exc)

WAVE_DIR = ROOT / "docs/waveforms"
PS_PER_US = 1_000_000  # VCD timestamps are in ps (timescale 1ns/1ps, vvp dumps ps)


def parse_vcd(path, wanted):
    """Return {full_name: [(t_ps, value_str), ...]} for the wanted full names."""
    id_to_names, data, scope = {}, {}, []
    with open(path) as f:
        defs, cur_t = True, 0
        for line in f:
            s = line.strip()
            if defs:
                if s.startswith("$scope"):
                    scope.append(s.split()[2])
                elif s.startswith("$upscope"):
                    if scope:
                        scope.pop()
                elif s.startswith("$var"):
                    p = s.split()
                    full = ".".join(scope + [p[4]])
                    if full in wanted:
                        id_to_names.setdefault(p[3], []).append(full)
                        data[full] = []
                elif s.startswith("$enddefinitions"):
                    defs = False
            elif s:
                if s[0] == "#":
                    cur_t = int(s[1:])
                elif s[0] in "01xzXZ" and s[1:] in id_to_names:
                    for full in id_to_names[s[1:]]:
                        data[full].append((cur_t, s[0]))
                elif s[0] in "bB":
                    val, ident = s[1:].split()
                    for full in id_to_names.get(ident, []):
                        data[full].append((cur_t, val))
    return data


def val_at(changes, t):
    i = bisect_right([c[0] for c in changes], t) - 1
    return changes[i][1] if i >= 0 else changes[0][1]


def edges(changes, rising=True):
    out, prev = [], None
    for t, v in changes:
        if prev is not None and ((rising and prev != "1" and v == "1") or
                                 (not rising and prev == "1" and v != "1")):
            out.append(t)
        prev = v
    return out


def draw(path, rows, data, t0, t1, title, shades=None, notes=None):
    """rows: list of (full_name, label, kind) with kind in {'bit','bus'}."""
    n = len(rows)
    fig, ax = plt.subplots(figsize=(11, 0.62 * n + 1.4))
    x0, x1 = t0 / PS_PER_US, t1 / PS_PER_US
    for a, b in (shades or []):
        ax.axvspan(a / PS_PER_US, b / PS_PER_US, color="#ffe8b3", zorder=0)
    for r, (full, label, kind) in enumerate(rows):
        base = (n - 1 - r) * 1.6
        ch = data[full]
        pts = [t0] + [t for t, _ in ch if t0 < t < t1] + [t1]
        if kind == "bit":
            xs = [t / PS_PER_US for t in pts]
            ys = [base + {"0": 0.0, "1": 1.0}.get(val_at(ch, t), 0.5) for t in pts]
            ax.step(xs, ys, where="post", color="#1f4e8c", lw=1.6)
        else:
            for a, b in zip(pts[:-1], pts[1:]):
                v = val_at(ch, a)
                try:
                    lbl = str(int(v, 2))
                except ValueError:
                    lbl = v
                xa, xb = a / PS_PER_US, b / PS_PER_US
                ax.add_patch(Rectangle((xa, base + 0.12), xb - xa, 0.76,
                                       fill=False, edgecolor="#1f4e8c", lw=1.2))
                if (xb - xa) > (x1 - x0) * 0.015:
                    ax.text((xa + xb) / 2, base + 0.5, lbl, ha="center",
                            va="center", fontsize=8, color="#1f4e8c")
        ax.text(x0 - (x1 - x0) * 0.012, base + 0.4, label, ha="right",
                va="center", fontsize=9, family="monospace")
    for xpos, txt, col in (notes or []):
        ax.annotate(txt, xy=(xpos / PS_PER_US, 0), xytext=(xpos / PS_PER_US, -1.1),
                    ha="center", fontsize=8.5, color=col,
                    arrowprops=dict(arrowstyle="->", color=col, lw=1.1))
    ax.set_xlim(x0, x1)
    ax.set_ylim(-1.5, (n - 1) * 1.6 + 1.3)
    ax.set_yticks([])
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.set_xlabel("time (µs)")
    ax.set_title(title, fontsize=11, fontweight="bold", loc="left")
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=130, facecolor="white", metadata={"Software": None})
    plt.close(fig)


if _HAVE_MPL:
    # Functionality: MISO is the bitwise inverse of orig_miso during the payload.
    p = "functionality_tb"
    sig = [f"{p}.SCK", f"{p}.NORM_CS_N", f"{p}.MISO",
           f"{p}.dut.u_recovered.orig_miso", f"{p}.dut.u_recovered.payload_active"]
    d = parse_vcd(BUILD / "functionality.vcd", set(sig))
    pa = edges(d[f"{p}.dut.u_recovered.payload_active"], True)[0]
    draw(WAVE_DIR / "functionality.png",
         [(f"{p}.SCK", "SCK", "bit"), (f"{p}.NORM_CS_N", "CS_N", "bit"),
          (f"{p}.dut.u_recovered.orig_miso", "orig_miso", "bit"),
          (f"{p}.MISO", "MISO (out)", "bit"),
          (f"{p}.dut.u_recovered.payload_active", "payload", "bit")],
         d, pa - 25000, pa + 200_000_000,
         "Functionality Trojan - payload transaction: MISO = NOT orig_miso (0xA6F03C59 -> 0x590FC3A6)",
         notes=[(pa + 10_000_000, "payload active: every MISO bit inverted", "#b22222")])

    # DoS: BUSY/ICE_LED/trigger latch high, persist with no clock, cleared by reset.
    p = "dos_tb"
    sig = [f"{p}.SCK", f"{p}.NORM_CS_N", f"{p}.BUSY", f"{p}.ICE_LED",
           f"{p}.RST_N", f"{p}.dut.u_recovered.trigger"]
    d = parse_vcd(BUILD / "dos.vcd", set(sig))
    trig = edges(d[f"{p}.dut.u_recovered.trigger"], True)[0]
    rec = [t for t in edges(d[f"{p}.RST_N"], False) if t > trig][0]
    draw(WAVE_DIR / "dos.png",
         [(f"{p}.SCK", "SCK", "bit"), (f"{p}.NORM_CS_N", "CS_N", "bit"),
          (f"{p}.dut.u_recovered.trigger", "trigger", "bit"),
          (f"{p}.BUSY", "BUSY", "bit"), (f"{p}.ICE_LED", "ICE_LED", "bit"),
          (f"{p}.RST_N", "RST_N", "bit")],
         d, trig - 70_000_000, rec + 45_000_000,
         "DoS Trojan - 0xBAADF00D latches BUSY/ICE_LED high; persists with no clock; cleared by reset",
         shades=[(trig, rec)],
         notes=[(trig, "trigger word complete", "#b22222"), (rec, "reset", "#2e7d32")])

    # Performance: core orig_busy ends at cycle 3; external BUSY extended to 11.
    p = "performance_tb"
    sig = [f"{p}.SCK", f"{p}.START", f"{p}.BUSY",
           f"{p}.dut.u_recovered.orig_busy", f"{p}.dut.u_recovered.ext_cnt",
           f"{p}.dut.u_recovered.start_cnt"]
    d = parse_vcd(BUILD / "performance.vcd", set(sig))
    op4 = edges(d[f"{p}.START"], True)[3]
    draw(WAVE_DIR / "performance.png",
         [(f"{p}.SCK", "SCK", "bit"), (f"{p}.START", "START", "bit"),
          (f"{p}.dut.u_recovered.start_cnt", "start_cnt", "bus"),
          (f"{p}.dut.u_recovered.orig_busy", "orig_busy", "bit"),
          (f"{p}.BUSY", "BUSY (out)", "bit"),
          (f"{p}.dut.u_recovered.ext_cnt", "ext_cnt", "bus")],
         d, op4 - 25_000_000, op4 + 265_000_000,
         "Performance Trojan - 4th START: core BUSY ends at cycle 3, external BUSY extended 8 cycles to 11",
         notes=[(op4, "4th START", "#b22222")])

    for _png in ("functionality.png", "dos.png", "performance.png"):
        display(Image(filename=str(WAVE_DIR / _png)))
    print("Waveforms written to", WAVE_DIR.relative_to(ROOT))

## 8. Results summary

Collect every case, persist the evidence, and assert the whole suite passed.

In [ ]:
order = ["clean_core.normal",
         "functionality.normal", "functionality.attack",
         "dos.normal", "dos.attack",
         "performance.normal", "performance.attack"]

print("Trojan verification results")
print("=" * 44)
for _label in order:
    print(f"  {'PASS' if RESULTS[_label]['ok'] else 'FAIL':4s}  {_label}")
all_pass = all(RESULTS[_l]["ok"] for _l in order)
print("=" * 44)
print("OVERALL:", "ALL PASS" if all_pass else "FAILURES PRESENT")

# Persist evidence. results/trojan_verification/ is tracked; *.vcd is git-ignored.
# Paths are made repo-relative so the committed transcript stays machine-neutral.
def _tidy(text):
    return text.replace(str(ROOT) + os.sep, "").replace(str(ROOT), ".")

EVID.joinpath("verification_log.txt").write_text("\n\n".join(
    f"===== {_l}: {'PASS' if RESULTS[_l]['ok'] else 'FAIL'} =====\n{_tidy(RESULTS[_l]['log'])}" for _l in order))
EVID.joinpath("summary.json").write_text(json.dumps({
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "repository": ROOT.name,
    "all_pass": all_pass,
    "cases": [{"label": _l, "pass": RESULTS[_l]["ok"]} for _l in order],
}, indent=2) + "\n")
print("Evidence written to", EVID.relative_to(ROOT))
assert all_pass, "One or more trojan verification cases failed"